In [29]:
import os, random
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset

from transformer import AutoTokenizer, AutoModelForCausalLM
from trl import SFTTrainer, SFTConfig

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

def build_dataset(tokenizer, train_size=100, val_size=100):
    def convert_to_prompt(example):
        instruction = example.get("instruction", "")
        context = example.get("context", "") or ""
        response = example.get("response", "")

        if context.strip():
            prompt = ("### Instruction: \n" f"{instruction} \n\n"
                      "### Context: \n" f"{context} \n\n"
                    "### Response: \n" f"{response} \n\n"
            )
        else:
            prompt = ("### Instruction: \n" f"{instruction} \n\n"
                    "### Response: \n" f"{response} \n\n"
            )

        text = prompt + response + tokenizer.eos_token
        return {"text": text} 
        

    ds = load_dataset("csv", datafiles={"train": train_path, "valid": valid_path})
    ds = load_dataset("databricks/databricks-dolly-15k")["train"]

    train_ds = ds["train"].select(range(val_size, val_size+train_size)
    val_ds = ds["train"].select(range(val_size)
        
    # train_ds = ds["train"].map(conver_to_prompt, remove_columns=ds["train"].column_names)
    # val_ds = ds["valid"].map(conver_to_prompt, remove_columns=val_ds.column_names)
    
    train_prom = train_ds.map(convert_to_prompt, remove_columns=train_ds.column_names)
    val_prom = val_ds.map(convert_to_prompt, remove_columns=val_ds.column_names)

    return train_prom, val_prom

def train_fn(model_name="distilgpt2", ouput_dir="./result"):
    set_seed(42)
    device = torch.device("cpu")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(model_name)
    model.to(device)

    train, val = build_dataset(tokenizer, train_size=100, val_size=100)

    args = SFTConfig(
        output_dir=output_dir,
        eval_strategy="epoch",
        num_train_epochs=1,
        per_dev
    )

    trainer = SFTTrainer(
        model=model,
        args=args,
        train_dataset=train,
        eval_dataset=val,
        processing_class=tokenizer
    )

In [5]:
ds[0].get("instruction", "")

'When did Virgin Australia start operating?'